# Experiment 1: Just-in-Time Retrieval vs Load-Everything

**The silly way:** dump ALL 7 papers into one call to answer one tiny question.
**The smart way:** show the agent a small *index*, let it load only the ONE paper it needs.

We measure both with **real** Bedrock token counts and see the difference.

## Setup

In [6]:
import json, time, os, sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# Find the project root whether we launch from notebooks/ or the repo root,
# and make `core` and `tools` importable.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/leapfrog/context_engineering-')

In [8]:
# Which Claude model + region to use (override in .env).
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-5-haiku-20241022-v1:0")
REGION   = os.getenv("AWS_REGION", "us-east-1")
print(MODEL_ID, "@", REGION)

us.anthropic.claude-3-5-haiku-20241022-v1:0 @ us-east-1


In [9]:
# The real Bedrock tokenizer (falls back to an estimate if no AWS creds yet).
from core.tokenizer import count_tokens
count_tokens("hello world hii my name is samir dahal ")

10

In [10]:
# Load the index (the "map") and the question cards.
INDEX = json.loads((PROJECT_ROOT / "data" / "index.json").read_text())


EVALS = json.loads((PROJECT_ROOT / "data" / "evals" / "exp01.json").read_text())
DEMO_QUERY = EVALS[0]["question"]

print(f"{len(INDEX)} papers | demo query: {DEMO_QUERY}")

7 papers | demo query: What do attention scores sum to, and why?


## Arm 1 - Naive (load everything)

Glue all 7 papers together and send them in one single Bedrock call.

**Heads up:** all 7 papers in full are ~227,000 tokens, which *overflows*
Claude's 200k context window. That overflow is itself the lesson - load-everything
literally does not fit. So we cap the blob to the most words that fit, and even
capped it dwarfs the smart approach.

In [11]:
# Glue every paper into one giant blob, and note each paper's real token size.
naive_blob = ""
naive_breakdown = {}
for paper in INDEX:
    text = (PROJECT_ROOT / paper["path"]).read_text(encoding="utf-8", errors="ignore")
    naive_blob += f"\n\n=== {paper['title']} ===\n{text}"
    naive_breakdown[paper["title"]] = count_tokens(text)


naive_breakdown




{'Attention Is All You Need': 7924,
 'Lost in the Middle': 12696,
 'RULER': 17156,
 'Retrieval-Augmented Generation': 12852,
 'A Survey on Hallucination in LLMs': 47554,
 'Indirect Prompt Injection': 23412,
 'RLHF effects on diversity': 22593}

In [12]:
# How heavy is the full blob? (~2 real tokens per word for dense papers)
print(f"{len(naive_blob.split()):,} words  ~= {len(naive_blob.split()) * 2:,} real tokens")
print("That overflows the 200k window for haiku model, so we cap it next.")

110,951 words  ~= 221,902 real tokens
That overflows the 200k window for haiku model, so we cap it next.


In [13]:
# Cap to the most words that fit the context window, then build the prompt.
NAIVE_MAX_WORDS = 97500        #
naive_text = " ".join(naive_blob.split()[:NAIVE_MAX_WORDS])
naive_prompt = f"{naive_text}\n\nQuestion: {DEMO_QUERY}\nAnswer:"

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
print(f"Capped naive context to {NAIVE_MAX_WORDS:,} words so it fits.")

Capped naive context to 97,500 words so it fits.


In [14]:
# ONE single Claude call. Time it.
t0 = time.time()
naive_resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": naive_prompt}]}],
    inferenceConfig={"maxTokens": 1000, "temperature": 0.0},
)
naive_latency = time.time() - t0
print(f"done in {naive_latency:.2f}s")

done in 53.19s


In [15]:
naive_resp

{'ResponseMetadata': {'RequestId': '8c649b13-420d-4126-b62a-2743ba904c84',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 10 Jun 2026 07:35:32 GMT',
   'content-type': 'application/json',
   'content-length': '1516',
   'connection': 'keep-alive',
   'x-amzn-requestid': '8c649b13-420d-4126-b62a-2743ba904c84'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': 'In the Transformer architecture described in the "Attention Is All You Need" paper, the attention scores for each query sum to 1 due to the softmax operation.\n\nSpecifically, in the scaled dot-product attention mechanism (Section 3.2.1), the attention scores are computed as follows:\n\n1. First, dot products are calculated between the query and all keys\n2. These dot products are scaled by dividing by √dk\n3. A softmax function is then applied to these scaled dot products\n\nThe softmax function ensures that the attention scores for a given query sum to 1. This has several impo

In [16]:
# Pull out the real answer and the real token usage.
naive_answer = naive_resp["output"]["message"]["content"][0]["text"]
naive_usage  = naive_resp["usage"]   # REAL: inputTokens / outputTokens / totalTokens
print(naive_answer[:300], "...")


In the Transformer architecture described in the "Attention Is All You Need" paper, the attention scores for each query sum to 1 due to the softmax operation.

Specifically, in the scaled dot-product attention mechanism (Section 3.2.1), the attention scores are computed as follows:

1. First, dot pr ...


In [17]:
naive_tokens = naive_usage["inputTokens"]
print(f"Naive input tokens: {naive_tokens:,}")

Naive input tokens: 200,250


## Arm 2 - Just-in-Time Retrieval

The agent sees only a tiny *menu* of papers, then uses `load_document`
to open the ONE it needs.

In [18]:
# The lightweight index the agent sees - titles + one-line descriptions only.
index_prompt = "Available research papers:\n"
for p in INDEX:
    index_prompt += f"[{p['id']}] {p['title']} - {p['desc']}\n"
print(index_prompt)

Available research papers:
[1706.03762] Attention Is All You Need - transformer architecture; self-attention; attention scores sum to 1.0
[2307.03172] Lost in the Middle - how LLMs use long contexts; U-shaped position performance curve
[2404.06654] RULER - measuring the real effective context length of long-context models
[2005.11401] Retrieval-Augmented Generation - RAG for knowledge-intensive NLP tasks
[2311.05232] A Survey on Hallucination in LLMs - taxonomy of hallucination including factuality vs faithfulness
[2302.12173] Indirect Prompt Injection - compromising LLM apps via malicious instructions in retrieved data
[2310.06452] RLHF effects on diversity - how RLHF reduces output diversity in language models



In [19]:
# How tiny is the menu vs the giant blob?
print(f"index: ~{count_tokens(index_prompt):,} tokens  (blob was ~{sum(naive_breakdown.values()):,})")

index: ~126 tokens  (blob was ~144,187)


In [20]:
# The agent's instructions.
system_prompt = f"""You are a research assistant with access to AI/ML papers.
{index_prompt}
When answering a question:
1. Identify which paper contains the answer
2. Call load_document with that paper's ID
3. Read it and answer with a citation

Be precise. Cite the paper ID in your answer."""

In [21]:
# Build the agent: Bedrock Claude + the load_document "hand".
from strands import Agent
from strands.models import BedrockModel
from tools.load_document import load_document

model = BedrockModel(model_id=MODEL_ID, region_name=REGION,
                     temperature=0.0, max_tokens=1000)
agent = Agent(model=model, tools=[load_document], system_prompt=system_prompt)

In [22]:
# Run it. The agent picks a paper, loads only that one, then answers.
t0 = time.time()
result = agent(DEMO_QUERY)
progressive_latency = time.time() - t0
print(f"\n done in {progressive_latency:.2f}s")

Let me retrieve the details from the Attention Is All You Need paper.
Tool #1: load_document
Based on the paper, attention scores are computed using a softmax function, which ensures they sum to 1.0. Specifically, in Section 3.2.1 (Scaled Dot-Product Attention), the authors describe the attention computation as:

Attention(Q, K, V) = softmax(QKT / √dk) V

The softmax function naturally normalizes the attention scores so that they sum to 1.0 across all positions. This normalization serves several purposes:
1. It converts raw compatibility scores into a probability distribution
2. It ensures each query (position) has a total attention weight of 1.0 across all keys
3. It allows the model to flexibly allocate attention across different input positions

The authors also note they scale the dot products by 1/√dk to prevent the softmax from entering regions with extremely small gradients when the dimension gets large.

The paper provides a detailed explanation in Section 3.2.1, specifically n

In [23]:
# The answer text.
progressive_answer = "".join(b.get("text", "") for b in result.message["content"] if "text" in b)
print(progressive_answer[:300], "...")

Based on the paper, attention scores are computed using a softmax function, which ensures they sum to 1.0. Specifically, in Section 3.2.1 (Scaled Dot-Product Attention), the authors describe the attention computation as:

Attention(Q, K, V) = softmax(QKT / √dk) V

The softmax function naturally norm ...


In [24]:
# Which document(s) did the agent actually open? Scan its transcript.
doc_ids = []
for m in agent.messages:
    for b in m.get("content", []):
        if isinstance(b, dict) and "toolUse" in b and b["toolUse"]["name"] == "load_document":
            doc_ids.append(b["toolUse"]["input"]["doc_id"])
doc_loaded = doc_ids[0] if doc_ids else "NONE"
doc_loaded

'1706.03762'

In [25]:
# The REAL tokens the agent used across the whole loop.
progressive_tokens = result.metrics.accumulated_usage["inputTokens"]
print(f"Progressive input tokens: {progressive_tokens:,}")

Progressive input tokens: 12,754


## Compare the two arms

In [26]:
import pandas as pd
pd.DataFrame([
    {"Arm": "Naive (all 7)",   "Input tokens": naive_tokens,       "Latency (s)": round(naive_latency, 2),       "Doc loaded": "ALL 7",    "Cited source": "no"},
    {"Arm": "Progressive",     "Input tokens": progressive_tokens, "Latency (s)": round(progressive_latency, 2), "Doc loaded": doc_loaded, "Cited source": "yes"},
])

,Arm,Input tokens,Latency (s),Doc loaded,Cited source
0,Naive (all 7),200250,53.19,ALL 7,no
1,Progressive,12754,11.60,1706.03762,yes


In [27]:
print(f"Token reduction: {naive_tokens / progressive_tokens:.1f}x fewer tokens")

Token reduction: 15.7x fewer tokens


In [28]:
# Token X-ray: where did every token go?
def xray(label, breakdown):
    total = sum(breakdown.values())
    print(f"\n{label} - {total:,} tokens")
    for src, tok in breakdown.items():
        pct = tok / total * 100 if total else 0
        print(f"  {src:<28} {'#' * int(pct/2):<50} {tok:>7,} ({pct:.0f}%)")

In [29]:
xray("Naive", naive_breakdown)

progressive_breakdown = {
    "system+index": count_tokens(system_prompt),
    "document":     count_tokens((PROJECT_ROOT / "data" / "corpus" / f"{doc_loaded}.txt").read_text(errors="ignore")) if doc_loaded != "NONE" else 0,
    "question":     count_tokens(DEMO_QUERY),
}
xray("Progressive", progressive_breakdown)


Naive - 144,187 tokens
  Attention Is All You Need    ##                                                   7,924 (5%)
  Lost in the Middle           ####                                                12,696 (9%)
  RULER                        #####                                               17,156 (12%)
  Retrieval-Augmented Generation ####                                                12,852 (9%)
  A Survey on Hallucination in LLMs ################                                    47,554 (33%)
  Indirect Prompt Injection    ########                                            23,412 (16%)
  RLHF effects on diversity    #######                                             22,593 (16%)

Progressive - 8,119 tokens
  system+index                 #                                                      185 (2%)
  document                     ################################################     7,924 (98%)
  question                                                                       

## What this proved

1. **Attention budget is finite.** All 7 papers = ~144k irrelevant tokens eating the budget. Attention thins across all of them.
2. **Just-in-Time retrieval stays lean.** Index = ~185 tokens → agent picks one paper → loads only that.
3. **References, not full data (Section 7.5).** The index is the map; `load_document` fetches the territory on demand.
4. **Quality AND efficiency.** The accuracy check below shows whether the focused context also produces a better answer — not just a cheaper one.

**Next:** Experiment 2 (Compaction) — when history MUST grow, summarize it at a threshold so the task survives past the window limit.

## Accuracy check — does just-in-time retrieval answer BETTER?

The research claim (Section 4.2): fewer, focused tokens → more attention on what matters → **better answer quality**, not just lower cost.

We test it by checking whether both answers contain the expected facts from `data/evals/exp01.json`.

In [ ]:
def contains_all_facts(answer: str, facts: list[str]) -> bool:
    """True if every expected fact appears in the answer (case-insensitive)."""
    lowered = answer.lower()
    return all(fact.lower() in lowered for fact in facts)


expected_facts = EVALS[0]["expected_facts"]
source_paper   = EVALS[0]["source_paper"]

naive_correct       = contains_all_facts(naive_answer, expected_facts)
progressive_correct = contains_all_facts(progressive_answer, expected_facts)
print("nive answer--------------")
print(naive_answer[:300])

print("progressive answer-----------------")
print(progressive_answer[:300])

print(f"Question:       {DEMO_QUERY}")
print(f"Expected facNAIVE_MAX_WORDSts: {expected_facts}")
print(f"Source paper:   {source_paper}")
print()
print(f"Naive answer       {'✅' if naive_correct else '❌'}  ({naive_tokens:,} tokens,       {naive_latency:.1f}s)")
print(f"Progressive answer {'✅' if progressive_correct else '❌'}  ({progressive_tokens:,} tokens, {progressive_latency:.1f}s)")

nive answer--------------
In the Transformer architecture described in the "Attention Is All You Need" paper, the attention scores for each query sum to 1 due to the softmax operation.

Specifically, in the scaled dot-product attention mechanism (Section 3.2.1), the attention scores are computed as follows:

1. First, dot pr
progressive answer-----------------
Based on the paper, attention scores are computed using a softmax function, which ensures they sum to 1.0. Specifically, in Section 3.2.1 (Scaled Dot-Product Attention), the authors describe the attention computation as:

Attention(Q, K, V) = softmax(QKT / √dk) V

The softmax function naturally norm
Question:       What do attention scores sum to, and why?
Expected facts: ['1.0', 'softmax']
Source paper:   1706.03762

Naive answer       ❌  (200,250 tokens,       53.2s)
Progressive answer ✅  (12,754 tokens, 11.6s)
